# 304: Post-process (calibrate) raw GSAT

Run this **once**, after the full production batch (`301_run_magicc.ipynb` via
`302_running_helpers.py`, across every shard) **and** any local backfill
(`303_run_magicc-missing.ipynb`) are both complete - i.e. after `individual_runs_raw/`
holds each scenario's *complete* ensemble (or as complete as it's going to get; see the
final summary cell below for anything still short).

This applies `CMIP7ScenarioMIPPostProcessor` - matching each ensemble's median over the
assessment period to the AR6/AR7 assessed historical warming - exactly **once** per
`(model, scenario, flag)`, over the complete raw ensemble, writing the calibrated result
to `individual_runs/` (the location downstream analysis, starting with
`500_calculate_peak_2100_warming.py`, already expects - no changes needed there).

This step is deliberately separate from `301`/`303` rather than folded into either of
them: `CMIP7ScenarioMIPPostProcessor`'s calibration shift is a single scalar computed
from the *median across whichever ensemble members are present* in the call that
computes it (see `notes/` for the full trace through `gcages`' source -
`get_temperatures_in_line_with_assessment` in `gcages/ar6/post_processing.py`).
Post-processing a partial batch (e.g. the main cluster run before backfill, or a
backfilled subset on its own) would each get their own, slightly different calibration
shift - baking a spurious systematic offset into the merged file. Running it once, here,
over the final complete ensemble avoids that entirely.

In [ ]:
import os
from pathlib import Path

import dotenv
import pandas as pd
import pandas_indexing as pix
from tqdm import tqdm

from gcages.cmip7_scenariomip.post_processing import CMIP7ScenarioMIPPostProcessor

dotenv.load_dotenv()
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

MAGICC_FLAGS = ("ANTHROPOGENIC", "CO2", "GHG")

In [ ]:
from utils import sanitize_label

batch_df = pd.read_csv(OUTPUT_FOLDER / "300_all_batches_v0.6.csv", index_col=0)
print(f"{len(batch_df)} scenarios in the batch file")

raw_dir = OUTPUT_FOLDER / "individual_runs_raw"
output_dir = OUTPUT_FOLDER / "individual_runs"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
post_processor = CMIP7ScenarioMIPPostProcessor.from_cmip7_scenariomip_config()

results_summary = []

for row in tqdm(list(batch_df.itertuples()), desc="Calibrating scenarios"):
    model, scenario = row.MODEL, row.SCENARIO
    sanitized_model = sanitize_label(model)
    sanitized_scenario = sanitize_label(scenario)

    for flag in MAGICC_FLAGS:
        raw_path = raw_dir / f"{sanitized_model}_{sanitized_scenario}_{flag}.csv"
        if not raw_path.exists():
            results_summary.append({
                "model": model, "scenario": scenario, "flag": flag,
                "status": "missing_raw_file", "n_members": 0,
            })
            continue

        raw = pd.read_csv(raw_path)
        id_cols = [c for c in raw.columns if not c.isdigit()]
        raw = raw.set_index(id_cols)
        raw.columns = raw.columns.astype(int)

        post_processed = post_processor(raw)
        result_df = post_processed.timeseries_run_id

        output_path = output_dir / f"{sanitized_model}_{sanitized_scenario}_{flag}.csv"
        result_df.to_csv(output_path)

        results_summary.append({
            "model": model, "scenario": scenario, "flag": flag,
            "status": "ok", "n_members": result_df.index.get_level_values("run_id").nunique(),
        })

summary_df = pd.DataFrame(results_summary)

In [ ]:
print(f"Calibrated {len(summary_df)} (scenario, flag) group(s)")
print(summary_df["status"].value_counts())
print()
print(f"Member count distribution (expect 600 for the vast majority):")
print(summary_df.loc[summary_df["status"] == "ok", "n_members"].value_counts().sort_index())

## Sanity check: anything still incomplete?

Cross-reference against the missing-members manifest (if it exists) - any
`(model, scenario, flag)` still short of 600 members here either wasn't in the manifest
at all (a `missing_raw_file` above - a genuine gap in the raw output, worth
investigating directly) or was backfilled but some members failed again even locally
(logged as a `WARNING` during `303`'s Part B run) and still needs manual follow-up.

In [ ]:
manifest_path = OUTPUT_FOLDER / "missing_members_manifest.csv"
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    still_incomplete = summary_df[
        (summary_df["status"] == "missing_raw_file")
        | ((summary_df["status"] == "ok") & (summary_df["n_members"] < 600))
    ]
    print(f"{len(still_incomplete)} (scenario, flag) group(s) still incomplete after calibration:")
    still_incomplete
else:
    print("No missing_members_manifest.csv found - either nothing was ever missing, "
          "or Part A of 303_run_magicc-missing.ipynb hasn't been run yet.")